# ADoTA metrics quick start

| Metric in the paper | Function (file under `src/`) |
|---|---|
| RMSE | `calculate_rmse` (`metrics/classic.py`) |
| MAPE | `calculate_pure_mape` (`metrics/classic.py`) |
| relative dose error $\rho$ | `calculate_relative_dose_error` (`metrics/classic.py`) |
| gamma pass rate $\Gamma$(dose %, DTA mm, cutoff %) | `gamma_index` (`metrics/gamma_pass_rate.py`); several criteria at once: `plan_gamma` (`metrics/plan_gamma.py`) |

Needs numpy, torch, pandas, PyYAML and, for the gamma functions, `pymedphys==0.41.0` with `numba` (install commands: `README.md`).

## Setup

In [ ]:
import os, sys, warnings, pathlib

# keep the thread pools small (shared login nodes limit the number of threads); must run before numpy / numba are imported
N_THREADS = str(min(4, os.cpu_count() or 1))
os.environ.setdefault("OMP_NUM_THREADS", N_THREADS)
os.environ.setdefault("NUMBA_NUM_THREADS", N_THREADS)

import numpy as np, pandas as pd, torch, pymedphys

# make `src/` importable wherever the notebook is started from
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p /"scripts" /"adota_metrics" /"src" / "metrics" / "classic.py").exists()), None)
assert ROOT is not None, "start Jupyter in the folder that contains src/"
sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore", message="Parameter `quiet`")      # pymedphys objects to a key that ADoTA's default gamma parameters still carry

from scripts.adota_metrics.src.metrics.classic import calculate_rmse, calculate_pure_mape, calculate_relative_dose_error
from scripts.adota_metrics.src.metrics.gamma_pass_rate import gamma_index, gamma_index_torch
from scripts.adota_metrics.src.metrics.plan_gamma import plan_gamma
from scripts.adota_metrics.src.adota.config import DEFAULT_GAMMA_PARAMS

print(f"numpy {np.__version__} | torch {torch.__version__} | pymedphys {pymedphys.__version__} | metric code from {ROOT.name}/src")

## Random input

A reference (the ground truth; in ADoTA the Monte Carlo dose) and a prediction: float numpy arrays of the same shape, `(depth, H, W)`.
ADoTA's model grid is (160, 30, 30) voxels of 2 mm; this demo uses a smaller cube because the gamma index gets slow when many voxels fail.
Except for `gamma_index_torch` (end of section 3), the functions take numpy arrays, so a network output goes through `.detach().cpu().numpy()` first.

In [ ]:
torch.manual_seed(0)
SHAPE = (48, 16, 16)                                                # (depth, H, W); ADoTA: (160, 30, 30)

reference = torch.rand(SHAPE).numpy()                               # random "reference dose", uniform in [0, 1)
prediction = np.clip(reference + 0.05 * torch.randn(SHAPE).numpy(), 0, None)    # the reference plus Gaussian noise (sigma 0.05), no negative dose
# with a real model: prediction = model(...).detach().cpu().numpy()
print(reference.shape, reference.dtype, prediction.dtype)

## RMSE, MAPE and $\rho$

Each function takes `(predicted, reference)` and returns one number. MAPE divides by its **second** argument, so the order matters,
and it has no threshold of its own: the paper (Eq. 8) averages only over voxels where the reference is above 10 % of its maximum, and the caller builds that mask.

In [ ]:
mask = reference > 0.10 * reference.max()                           # voxels above 10 % of the reference maximum

rmse     = calculate_rmse(prediction, reference)
mape_all = calculate_pure_mape(prediction, reference)               # every voxel where the reference is not 0
mape_10  = calculate_pure_mape(prediction[mask], reference[mask])   # as in the paper (Eq. 8)
rho      = calculate_relative_dose_error(prediction, reference)

print(f"RMSE                 {rmse:.4f}   (in the unit of the arrays)")
print(f"MAPE, all voxels     {mape_all:.2f} %")
print(f"MAPE, ref > 10 % max {mape_10:.2f} %   ({mask.sum()} of {mask.size} voxels)")
print(f"rho                  {rho:.3f} %")

## Gamma pass rate

`gamma_index(ground_truth, prediction, scale, gamma_params, resolution)` takes the **reference first**. The criterion sits in `gamma_params`;
ADoTA's default, printed below, is $\Gamma$(2 %, 2 mm, 10 %): a dose tolerance of 2 % of the reference maximum (global gamma), a distance-to-agreement of 2 mm,
and only voxels where the reference is at least 10 % of its maximum are evaluated.
The three numbers of the criterion are `dose_percent_threshold`, `distance_mm_threshold` and `lower_percent_dose_cutoff`; `interp_fraction=10` makes the search step a tenth of the distance-to-agreement,
`max_gamma` caps the gamma values, `random_subset=None` evaluates every voxel, and `quiet` is a deprecated key that pymedphys only warns about.

In [ ]:
DEFAULT_GAMMA_PARAMS

**Calling it.** `resolution` is the voxel size in mm per axis (the default is `(1, 1, 1)`; the paper's grid is 2 x 2 x 2 mm). Of `scale` the function reads only `y_max`, and only for its own optional `cutoff` argument (default 0), but the key has to be there.
`gamma_index` sets voxels below `cutoff` % of `y_max` (with the default 0: the negative ones) to 0 *in the arrays you pass*, so pass copies.
It returns the gamma map (NaN, i.e. not evaluated, becomes 0) and two pass rates as fractions, not percent: `[0]` is the one `run_model.py` and `plan_gamma` report, it counts only voxels with gamma above 0 and so ignores voxels that agree exactly;
`[1]` is the standard passing / evaluated voxels. The two differ only when evaluated voxels agree exactly.

In [ ]:
RES = (2.0, 2.0, 2.0)                                               # mm per voxel (paper: 2 x 2 x 2 mm)
scale = {"y_min": 0.0, "y_max": float(reference.max())}

gamma_map, rates = gamma_index(reference.copy(), prediction.copy(), scale, DEFAULT_GAMMA_PARAMS, RES)
print(f"gamma pass rate: [0] {rates[0]:.2%} | [1] {rates[1]:.2%}")
print(f"gamma map {gamma_map.shape}: {np.count_nonzero(gamma_map > 0)} voxels with gamma > 0, {np.count_nonzero(gamma_map > 1)} failing (gamma > 1), largest gamma {gamma_map.max():.2f}")

**Several criteria at once.** `plan_gamma` runs the gamma index for a list of `(dose %, DTA mm, cutoff %)` triples, the notation of the paper's tables, and reports `[0]` in percent.
Mind the argument order: here the **prediction comes first**. Each result also holds its gamma map (`result["gamma_map"]`).

In [ ]:
criteria = [(2, 2, 10), (1, 3, 0.1)]                                # (dose %, DTA mm, cutoff %)
results = plan_gamma(prediction, reference, RES, criteria)          # (dose_eval, dose_ref, voxel size in mm, criteria)
pd.Series({r["label"]: r["pass_rate_pct"] for r in results}, name="gamma pass rate [%]").round(2).to_frame()

**With torch tensors.** `run_model.py` calls `gamma_index_torch`. It takes normalised `(1, 1, D, H, W)` tensors (batch size 1, one channel),
de-normalises them with `scale` (`dose = tensor * (y_max - y_min) + y_min`) and calls `gamma_index`. Here the random cubes are normalised by `y_max`, which gives the same pass rates as above.

In [ ]:
scale_t = {"y_min": 0.0, "y_max": float(reference.max())}          # run_model.py: {"y_min": scale["min_ds"], "y_max": scale["max_ds"]}
to_tensor = lambda a: torch.from_numpy(a / scale_t["y_max"])[None, None]    # normalise to [0, 1], add the (batch, channel) dimensions

_, rates_t = gamma_index_torch(to_tensor(reference), to_tensor(prediction), scale_t, DEFAULT_GAMMA_PARAMS, RES)
print(f"gamma pass rate via gamma_index_torch: [0] {rates_t[0]:.2%} | [1] {rates_t[1]:.2%}")

## Keep in mind

- **Inputs:** equally shaped float numpy arrays, `(depth, H, W)`; a network output goes through `.detach().cpu().numpy()` first.
- **MAPE:** call it as `(prediction, reference)` with the mask built on the reference. Several scripts in the ADoTA repository pass the arguments the other way round.
- **Gamma:** `gamma_index(reference, prediction, ...)` but `plan_gamma(prediction, reference, ...)`; `resolution` is in mm; pass rates are fractions.
  `gamma_index` changes the arrays you pass (negative voxels become 0), so give it copies. `[0]` raises `ZeroDivisionError` when no voxel has gamma above 0 (an exactly perfect prediction), and the run time grows with the number of failing voxels.
- **RMSE** is the only metric with a unit, and it comes out in the unit of the arrays you pass. The repository stores dose in eV/g per proton (`data/README.md` upstream), but `to_gy` (`src/utils/unit_conversions.py`) is written for MeV/g:
  applied to such data it gives Gy per 10^6 protons, while the paper reports RMSE in Gy per 10^9 particles (a factor 1000). MAPE, $\rho$ and the gamma pass rate do not depend on the unit.

# Metric behavior study

This notebook isolates the behavior of the ADoTA metrics under controlled synthetic errors.
The purpose is to understand:

- what each metric measures
- what it is sensitive to
- how it reacts to bias, scaling, shift, and local hotspot errors
- how the gamma pass rate behaves in the exact-perfect case

In [ ]:
def safe_gamma(ref, pred):
    try:
        _, rates = gamma_index(
            ref.copy(),
            pred.copy(),
            {"y_min": 0.0, "y_max": float(ref.max())},
            DEFAULT_GAMMA_PARAMS,
            (2.0, 2.0, 2.0),
        )
        return rates[0], rates[1]
    except ZeroDivisionError:
        return np.nan, 1.0


In [ ]:
def metric_table(ref, pred):
    mask = ref > 0.10 * ref.max()
    rmse = calculate_rmse(pred, ref)
    mape_all = calculate_pure_mape(pred, ref)
    mape_10 = calculate_pure_mape(pred[mask], ref[mask])
    rho = calculate_relative_dose_error(pred, ref)

    g0, g1 = safe_gamma(ref, pred)

    return {
        "RMSE": rmse,
        "MAPE_all": mape_all,
        "MAPE_10pct": mape_10,
        "rho": rho,
        "gamma_pass_rate": g0,
        "gamma_pass_rate_standard": g1,
    }


In [ ]:
# New focused diagnostic block for metric sensitivity

# 1) same RMSE, different error geometry
rng = np.random.default_rng(1)
ref = rng.random((32, 16, 16))

# broad low-amplitude offset
case_bias = ref + 0.12

# same shape, same overall voxel magnitude, but shifted
case_shift = np.roll(ref, 3, axis=0)

# local hotspot failure
case_hotspot = ref.copy()
case_hotspot[8:12, 4:8, 4:8] *= 0.4
case_hotspot = np.clip(case_hotspot, 0, None)

# build a comparison table
rows = []
for name, pred in {
    "bias": case_bias,
    "shift": case_shift,
    "hotspot": case_hotspot,
}.items():
    row = metric_table(ref, pred)
    row["case"] = name
    rows.append(row)

pd.DataFrame(rows).set_index("case").round(4)

# 2) gamma threshold sweep
criteria = [
    (2, 2, 10),
    (2, 3, 10),
    (1, 2, 10),
    (2, 2, 0),
]

# a shifted prediction should show larger effect under stricter gamma conditions
shifted = np.roll(ref, 2, axis=1)

results = []
for dose_pct, dta_mm, cutoff in criteria:
    params = dict(DEFAULT_GAMMA_PARAMS)
    params["dose_percent_threshold"] = dose_pct
    params["distance_mm_threshold"] = dta_mm
    params["lower_percent_dose_cutoff"] = cutoff
    _, rates = gamma_index(ref.copy(), shifted.copy(), {"y_min": 0.0, "y_max": float(ref.max())}, params, (2.0, 2.0, 2.0))
    results.append({
        "dose_percent": dose_pct,
        "dta_mm": dta_mm,
        "cutoff": cutoff,
        "pass_rate_0": rates[0],
        "pass_rate_1": rates[1],
    })

pd.DataFrame(results).round(4)


In [ ]:
# Each prediction starts from the same reference.
# Change one setting at a time and rerun this cell.

def shift_volume(volume, shift, fill_value=0.0):
    """Shift a 3D array without wrapping values around its edges."""
    shifted = np.full_like(volume, fill_value)

    source_slices = []
    destination_slices = []

    for size, offset in zip(volume.shape, shift):
        if abs(offset) >= size:
            return shifted

        if offset >= 0:
            source_slices.append(slice(0, size - offset))
            destination_slices.append(slice(offset, size))
        else:
            source_slices.append(slice(-offset, size))
            destination_slices.append(slice(0, size + offset))

    shifted[tuple(destination_slices)] = volume[tuple(source_slices)]
    return shifted


# Try changing these strengths
bias = 0.05
scale_factor = 1.10
noise_sigma = 0.03
shift_voxels = 1

predictions = {
    "perfect": reference.copy(),
    "bias": reference + bias,
    "scale": reference * scale_factor,
    "noise": np.clip(
        reference + np.random.default_rng(0).normal(0, noise_sigma, reference.shape),
        0,
        None,
    ),
    "shift": shift_volume(reference, (shift_voxels, 0, 0)),
}

rows = []
for case_name, prediction_case in predictions.items():
    row = metric_table(reference, prediction_case)
    row["case"] = case_name
    rows.append(row)

pd.DataFrame(rows).set_index("case").round(4)

In [ ]:
# Dose perturbation experiment: keep the metric functions and gamma settings fixed.
# These changes alter dose arrays; they do not simulate proton transport at a new energy.

absolute_change = 0.05 * float(reference.max())  # same dose amount added/subtracted per voxel
percent_change = 0.05                         # same percentage applied to each voxel
noise_sigma = 0.01 * float(reference.max())


def shift_depth(volume, voxels):
    """Shift along the depth axis without wrapping values around the volume."""
    shifted = np.zeros_like(volume)
    if voxels > 0:
        shifted[voxels:] = volume[:-voxels]
    elif voxels < 0:
        shifted[:voxels] = volume[-voxels:]
    else:
        shifted[:] = volume
    return shifted


rng = np.random.default_rng(0)
dose_variants = {
    "unchanged": reference.copy(),
    "absolute increase": reference + absolute_change,
    "absolute decrease": np.clip(reference - absolute_change, 0, None),
    "5% increase": reference * (1 + percent_change),
    "5% decrease": reference * (1 - percent_change),
    "added noise": np.clip(
        reference + rng.normal(0, noise_sigma, reference.shape), 0, None
    ),
    "depth shift by 1 voxel": shift_depth(reference, 1),
}

rows = []
for case_name, altered_dose in dose_variants.items():
    row = metric_table(reference, altered_dose)
    row["case"] = case_name
    rows.append(row)

perturbation_results = pd.DataFrame(rows).set_index("case")
perturbation_results.round(4)


## Controlled dose perturbations

This is a synthetic, smooth dose-shaped volume, not a physical proton simulation. Each row changes only the dose array; the metric functions and default gamma criterion stay fixed. The absolute-difference columns make the voxel-level perturbation size easy to compare with the metric scores.

In [ ]:
# Build a smooth, dose-shaped 3D reference for a more interpretable test.
# It is an illustrative shape, not a physically calculated proton dose.
depth, height, width = 48, 16, 16
spacing_mm = 2.0
z = np.arange(depth) * spacing_mm
y = (np.arange(height) - (height - 1) / 2) * spacing_mm
x = (np.arange(width) - (width - 1) / 2) * spacing_mm

z_peak_mm = z[depth // 2]
depth_profile = np.exp(-0.5 * ((z - z_peak_mm) / 12.0) ** 2)
y_grid, x_grid = np.meshgrid(y, x, indexing="ij")
transverse_profile = np.exp(-0.5 * (x_grid**2 + y_grid**2) / (10.0**2))
dose_reference = depth_profile[:, None, None] * transverse_profile[None, :, :]
dose_reference /= dose_reference.max()

# Perturb one property per case. Gamma parameters remain DEFAULT_GAMMA_PARAMS.
absolute_change = 0.05 * float(dose_reference.max())
percent_change = 0.05
noise_sigma = 0.01 * float(dose_reference.max())
rng = np.random.default_rng(0)


def shift_depth_without_wrap(volume, voxels):
    shifted = np.zeros_like(volume)
    if voxels > 0:
        shifted[voxels:] = volume[:-voxels]
    elif voxels < 0:
        shifted[:voxels] = volume[-voxels:]
    else:
        shifted[:] = volume
    return shifted


local_drop = dose_reference.copy()
local_drop[depth // 2 - 2:depth // 2 + 3, 5:11, 5:11] *= 0.70

perturbed_doses = {
    "unchanged": dose_reference.copy(),
    "absolute +5% of max": dose_reference + absolute_change,
    "absolute -5% of max": np.clip(dose_reference - absolute_change, 0, None),
    "relative +5%": dose_reference * (1 + percent_change),
    "relative -5%": dose_reference * (1 - percent_change),
    "noise, sigma 1% max": np.clip(
        dose_reference + rng.normal(0, noise_sigma, dose_reference.shape), 0, None
    ),
    "local peak -30%": local_drop,
    "depth shift +1 voxel": shift_depth_without_wrap(dose_reference, 1),
}

rows = []
for case_name, altered_dose in perturbed_doses.items():
    difference = np.abs(altered_dose - dose_reference)
    row = metric_table(dose_reference, altered_dose)
    row["mean_abs_difference"] = difference.mean()
    row["max_abs_difference"] = difference.max()
    row["case"] = case_name
    rows.append(row)

controlled_results = pd.DataFrame(rows).set_index("case")
controlled_results.round(4)


### Data Loader

In [2]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'


In [ ]:
import json
from monai.transforms import MapTransform, ResizeWithPadOrCropd , EnsureTyped,Compose, LoadImaged, EnsureChannelFirstd, Spacingd, CenterSpatialCropd
import nibabel as nib
import numpy as np
from monai.data import PersistentDataset, DataLoader, Dataset
import torch
from omegaconf import OmegaConf
import os
from pathlib import Path

from scripts.metaembedder import InjectGaussianBeamPriord, InjectEnergyDepositionFieldd, Ray_Info

from monai.transforms import ScaleIntensityRanged

from torch.utils.data import Dataset as TorchDataset

In [55]:
cfg = {
    "train_list": "/project/nr_dose/data/train.json",
    "val_list": "/project/nr_dose/data/val.json",
}

In [54]:
train_list = json.load(open(cfg["train_list"], "r"))
val_list = json.load(open(cfg["val_list"], "r"))

print("Training examples:", len(train_list))
print("Validation examples:", len(val_list))
print("Fields in one training example:", train_list[0].keys())
print("One training example:", train_list[0])

Training examples: 72359
Validation examples: 8640
Fields in one training example: dict_keys(['ct', 'gt_dose', 'ray_source', 'ray_target', 'condition'])
One training example: {'ct': '//scratch/nr_dose/training/1THB095/image/ct.mha', 'gt_dose': '//scratch/nr_dose/training/1THB095/dose/Dose_B0_R0_L0.mha', 'ray_source': [-37.25, -1043.89, 4.66], 'ray_target': [-37.25, -43.8900000000001, 4.66], 'condition': [0.0, 1.0, 41.3788]}


In [53]:
from pathlib import Path

example = train_list[0]

print("CT path:", example["ct"])
print("CT file exists:", Path(example["ct"]).is_file())

print("Dose path:", example["gt_dose"])
print("Dose file exists:", Path(example["gt_dose"]).is_file())

CT path: //scratch/nr_dose/training/1THB095/image/ct.mha
CT file exists: True
Dose path: //scratch/nr_dose/training/1THB095/dose/Dose_B0_R0_L0.mha
Dose file exists: True


In [52]:
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityRanged

one_example_transform = Compose([
    LoadImaged(keys=["ct", "gt_dose"]),
    EnsureChannelFirstd(keys=["ct", "gt_dose"]),
    ScaleIntensityRanged(
        keys=["ct"],
        a_min=cfg["ct_min"],
        a_max=cfg["ct_max"],
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
])

loaded_example = one_example_transform(example)

print("CT shape:", loaded_example["ct"].shape)
print("Dose shape:", loaded_example["gt_dose"].shape)
print("Condition:", loaded_example["condition"])

CT shape: torch.Size([1, 443, 440, 133])
Dose shape: torch.Size([1, 443, 440, 133])
Condition: [0.0, 1.0, 41.3788]


## crop transform from the reference and try it on this one patient.

In [57]:
import nibabel as nib
import numpy as np
import torch
from monai.transforms import MapTransform

class ExtractSlabsAroundZ(MapTransform):
    def __init__(
        self,
        keys,
        source_key="ray_source",
        slice_radius=15,
        allow_missing_keys=False,
    ):
        super().__init__(keys, allow_missing_keys)
        self.source_key = source_key
        self.slice_radius = slice_radius

    def __call__(self, data):
        d = dict(data)

        # The first 3 values are the beam's physical X, Y, Z position.
        physical_point = np.asarray(d[self.source_key][:3], dtype=float)

        for key in self.key_iterator(d):
            img = d[key]

            # Convert physical coordinates to voxel coordinates.
            affine = img.affine.cpu().numpy()
            inv_affine = np.linalg.inv(affine)
            voxel_coords = nib.affines.apply_affine(inv_affine, physical_point)

            # This crop uses the Z voxel index and keeps all X and Y voxels.
            z_index = int(np.round(voxel_coords[2]))
            z_start = z_index - self.slice_radius
            z_end = z_index + self.slice_radius + 1  # Python excludes the end

            # Clamp the slice range to the image boundary.
            z_min = max(0, z_start)
            z_max = min(img.shape[3], z_end)
            cropped_img = img[:, :, :, z_min:z_max]

            # Add zero slices if the crop extends beyond the image.
            pad_left = max(0, -z_start)
            pad_right = max(0, z_end - img.shape[3])

            if pad_left > 0 or pad_right > 0:
                cropped_img = torch.nn.functional.pad(
                    torch.as_tensor(cropped_img),
                    (pad_left, pad_right, 0, 0, 0, 0),
                    mode="constant",
                    value=0,
                )

            d[key] = cropped_img

        return d

In [20]:
crop_transform = ExtractSlabsAroundZ(
    keys=["ct", "gt_dose"],
    source_key="ray_source",
    slice_radius=15,
)

cropped_example = crop_transform(loaded_example)

print("CT shape after crop:", cropped_example["ct"].shape)
print("Dose shape after crop:", cropped_example["gt_dose"].shape)

CT shape after crop: torch.Size([1, 443, 440, 31])
Dose shape after crop: torch.Size([1, 443, 440, 31])


In [23]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Lambdad,
    Spacingd,
    ResizeWithPadOrCropd,
)

# Add the settings we found to your existing cfg dictionary.
cfg.update({
    "ct_min": -1024,
    "ct_max": 2000,
    "pixdim": [4.0, 4.0, 3.0],
    "roi_size": [128, 128, 32],
})

def scale_dose_by_1000(dose):
    return dose * 1000.0

one_example_transform = Compose([
    LoadImaged(keys=["ct", "gt_dose"]),
    EnsureChannelFirstd(keys=["ct", "gt_dose"]),

    # Match the reference: scale CT values and multiply dose values by 1000.
    ScaleIntensityRanged(
        keys=["ct"],
        a_min=cfg["ct_min"],
        a_max=cfg["ct_max"],
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    Lambdad(keys=["gt_dose"], func=scale_dose_by_1000),

    # Crop around ray_source, then standardize spacing and image size.
    ExtractSlabsAroundZ(
        keys=["ct", "gt_dose"],
        source_key="ray_source",
        slice_radius=15,
    ),
    Spacingd(
        keys=["ct", "gt_dose"],
        pixdim=cfg["pixdim"],
        mode="trilinear",
    ),
    ResizeWithPadOrCropd(
        keys=["ct", "gt_dose"],
        spatial_size=cfg["roi_size"],
    ),
])

prepared_example = one_example_transform(train_list[0])

print("CT shape:", prepared_example["ct"].shape)
print("Dose shape:", prepared_example["gt_dose"].shape)
print("CT value range:",
      prepared_example["ct"].min().item(),
      "to",
      prepared_example["ct"].max().item())
print("Condition:", prepared_example["condition"])

CT shape: torch.Size([1, 128, 128, 32])
Dose shape: torch.Size([1, 128, 128, 32])
CT value range: 0.0 to 0.9161367416381836
Condition: [0.0, 1.0, 41.3788]


### DATA LOADER

In [67]:
from scripts.metaembedder import (
    InjectGaussianBeamPriord,
    InjectEnergyDepositionFieldd,
    Ray_Info,
)


class SaveCTAffined(MapTransform):
    """Store the CT affine as a regular tensor so it survives PersistentDataset caching."""
    def __init__(self, keys=("ct",), allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            affine = d[key].affine.detach().cpu().numpy().copy()
            d[f"{key}_affine"] = torch.tensor(affine, dtype=torch.float64)
        return d


class RestoreCTAffined(MapTransform):
    """Rebuild the CT MetaTensor after it has been read back from the cache."""
    def __init__(self, keys=("ct",), allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            if not isinstance(d[key], MetaTensor):
                affine = d[f"{key}_affine"].detach().cpu()
                d[key] = MetaTensor(d[key], affine=affine)
        return d


def get_post_transforms(cfg):
    transforms = [RestoreCTAffined(keys=["ct"])]
    final_keys = ["ct", "gt_dose", "condition"]

    if cfg.get("inject_geometric_prior", False):
        transforms.append(
            InjectGaussianBeamPriord(
                keys=["geometric_prior"],
                source_key="ray_source",
                target_key="ray_target",
                ref_key="ct",
                sigma=cfg["sigma"],
                flip_lps_to_ras=True,
                prior_mode=cfg["prior_mode"],
            )
        )
        final_keys.append("geometric_prior")

    if cfg.get("inject_energy_field", False):
        transforms.append(
            InjectEnergyDepositionFieldd(
                keys=["field"],
                spacing=cfg["pixdim"],
            )
        )
        final_keys.append("field")

    if cfg.get("drop_priors", False):
        transforms.append(
            RandChannelDropoutd(
                keys=["geometric_prior", "field"],
                prob=0.15,
            )
        )

    if cfg.get("ray_info", False):
        transforms.append(Ray_Info(keys=["ray_source", "ray_target"]))
        final_keys.extend([
            "positions",
            "ray_z_positions",
            "ray_start_anchor",
            "ray_end_offset",
            "ray_bragg_offset",
        ])

    transforms.extend([
        EnsureTyped(keys=final_keys, track_meta=True, dtype=torch.float32),
        SelectItemsd(keys=final_keys),
    ])
    return Compose(transforms)


def get_loaders(cfg):
    with open(cfg["train_list"], "r") as f:
        train_list = json.load(f)
    with open(cfg["val_list"], "r") as f:
        val_list = json.load(f)

    train_transforms = Compose([
        LoadImaged(keys=["ct", "gt_dose"]),
        EnsureChannelFirstd(keys=["ct", "gt_dose"]),
        ScaleIntensityRanged(
            keys=["ct"],
            a_min=cfg["ct_min"],
            a_max=cfg["ct_max"],
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        Lambdad(keys=["gt_dose"], func=scale_dose_by_1000),
        ExtractSlabsAroundZ(
            keys=["ct", "gt_dose"],
            source_key="ray_source",
            slice_radius=15,
        ),
        Spacingd(
            keys=["ct", "gt_dose"],
            pixdim=cfg["pixdim"],
            mode="trilinear",
        ),
        ResizeWithPadOrCropd(
            keys=["ct", "gt_dose"],
            spatial_size=cfg["roi_size"],
        ),
        SaveCTAffined(keys=["ct"]),
    ])

    post_transforms = get_post_transforms(cfg)

    if cfg["dataset_type"] == "persistent":
        os.makedirs(cfg["cache_dir"], exist_ok=True)
        train_cached_ds = PersistentDataset(
            data=train_list,
            transform=train_transforms,
            cache_dir=cfg["cache_dir"],
        )
        val_cached_ds = PersistentDataset(
            data=val_list,
            transform=train_transforms,
            cache_dir=cfg["cache_dir"],
        )
        train_ds = Dataset(data=train_cached_ds, transform=post_transforms)
        val_ds = Dataset(data=val_cached_ds, transform=post_transforms)

    elif cfg["dataset_type"] == "dataset":
        all_transforms = Compose([train_transforms, post_transforms])
        train_ds = Dataset(data=train_list, transform=all_transforms)
        val_ds = Dataset(data=val_list, transform=all_transforms)

    else:
        raise ValueError(f"Unknown dataset_type: {cfg['dataset_type']}")

    val_ds = PatientIDWrapper(val_ds, val_list)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"],
        persistent_workers=cfg["persistent_workers"],
        prefetch_factor=cfg["prefetch_factor"],
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        persistent_workers=cfg["persistent_workers"],
        prefetch_factor=cfg["prefetch_factor"],
        pin_memory=True,
    )
    return train_loader, val_loader


# Use a new cache because the earlier entries don't contain ct_affine.
cfg["cache_dir"] = "/scratch/db/cache_dataloader_replica_v3"

train_loader, val_loader = get_loaders(cfg)

# Deliberately access the same item twice:
# first access creates its cache entry; second reads it back from disk.
sample_first = train_loader.dataset[0]
sample_from_cache = train_loader.dataset[0]

print("Cached-read CT type:", type(sample_from_cache["ct"]))
print("Cached-read CT has affine:", hasattr(sample_from_cache["ct"], "affine"))
print("CT shape:", tuple(sample_from_cache["ct"].shape))
print("Dose shape:", tuple(sample_from_cache["gt_dose"].shape))

Cached-read CT type: <class 'monai.data.meta_tensor.MetaTensor'>
Cached-read CT has affine: True
CT shape: (1, 128, 128, 32)
Dose shape: (1, 128, 128, 32)


In [66]:
train_loader, val_loader = get_loaders(cfg)

print("Training examples:", len(train_loader.dataset))
print("Validation examples:", len(val_loader.dataset))

Training examples: 72359
Validation examples: 8640


In [71]:
model_input = torch.cat(
    [one_batch["ct"], one_batch["geometric_prior"], one_batch["field"]],
    dim=1,
)
target = one_batch["gt_dose"]

print("Model input:", tuple(model_input.shape))
print("Dose target:", tuple(target.shape))

Model input: (1, 3, 128, 128, 32)
Dose target: (1, 1, 128, 128, 32)


In [70]:
from torch.utils.data import Subset

batch_check_loader = DataLoader(
    Subset(train_loader.dataset, [0]),
    batch_size=1,
    num_workers=0,
)

batch = next(iter(batch_check_loader))

model_input = torch.cat(
    [batch["ct"], batch["geometric_prior"], batch["field"]],
    dim=1,
)
target = batch["gt_dose"]

print("Model input:", tuple(model_input.shape))
print("Dose target:", tuple(target.shape))
print("Condition:", tuple(batch["condition"].shape))

Model input: (1, 3, 128, 128, 32)
Dose target: (1, 1, 128, 128, 32)
Condition: (1, 3)
